# Pipeline de données Cholera
Ce notebook fusionne les étapes de chargement, nettoyage, normalisation et import PostgreSQL en un seul workflow. Toutes les transformations sont effectuées localement avant l'insertion dans la base de données.

In [ ]:
from pathlib import Path
import os
import logging
import pandas as pd
from sqlalchemy import create_engine, text

ROOT = Path.cwd().resolve()
if not (ROOT / "db").exists():
    ROOT = ROOT.parent
DB_DIR = ROOT / "db"
DATA_DIR = ROOT / "data"
NOTEBOOK_LOGS = ROOT / "logs"
NOTEBOOK_LOGS.mkdir(exist_ok=True)

os.environ.setdefault("POSTGRES_USER", "bearing")
os.environ.setdefault("POSTGRES_PASSWORD", "Couspdata")
os.environ.setdefault("POSTGRES_DB", "ids_db")
os.environ.setdefault("POSTGRES_HOST", "localhost")
os.environ.setdefault("POSTGRES_PORT", "5432")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s: %(message)s")
logger = logging.getLogger("cholera_pipeline")
logger.info("Workspace root: %s", ROOT)


## Charger et fusionner les données
- Charger le fichier IDS depuis `db/ids.xlsx`
- Charger les fichiers LL depuis `db/`
- Préparer les DataFrames en mémoire avant toute insertion en base

In [ ]:
def load_ids():
    path = DB_DIR / "ids.xlsx"
    if not path.exists():
        raise FileNotFoundError(f"Fichier IDS introuvable: {path}")
    df = pd.read_excel(path, sheet_name="db", engine="openpyxl")
    df = df.rename(columns={
        "NUM": "code_zone", "PAYS": "pays", "PROV": "province", "ZS": "zone_sante",
        "POP": "population", "NUMSEM": "num_semaine", "DEBUTSEM": "debut_semaine_originale",
        "MALADIE": "maladie", "C328TNN": "cas_tnn", "DTNN": "deces_tnn",
        "C011MOIS": "cas_0_11_mois", "D011MOIS": "deces_0_11_mois",
        "C1259MOIS": "cas_12_59_mois", "D1259MOIS": "deces_12_59_mois",
        "C515ANS": "cas_5_15_ans", "D515ANS": "deces_5_15_ans",
        "CP15ANS": "cas_15_plus", "DP15ANS": "deces_15_plus",
        "TOTALCAS": "cas_total", "TOTALDECES": "deces_total",
        "LETAL": "letalite", "ATTAQ": "taux_attaque",
        "RecStatus": "rec_status", "UniqueKey": "unique_key", "ANNEE": "annee"
    })
    if "debut_semaine_originale" in df.columns and "debut_semaine" not in df.columns:
        df["debut_semaine"] = pd.to_datetime(df["debut_semaine_originale"], errors="coerce")
    return df

def load_ll():
    files = list(DB_DIR.glob("rdc_compilation*_LL_Cholera_*.xlsx"))
    if not files:
        fallback = DB_DIR / "rdc_compilation_LL_Cholera_*.xlsx"
        if fallback.exists():
            files = [fallback]
    if not files:
        raise FileNotFoundError("Aucun fichier LL trouvé dans db/")

    dfs = []
    for path in files:
        xl = pd.ExcelFile(path, engine="openpyxl")
        if "LL_Cholera" not in xl.sheet_names:
            logger.warning("Feuille LL_Cholera absente dans %s", path)
            continue
        df = pd.read_excel(path, sheet_name="LL_Cholera", engine="openpyxl")
        df.columns = [str(c).strip() for c in df.columns]
        rename_map = {
            "N_epid_prov": "n_epid_prov", "N_epid": "n_epid", "Statut_a_l_arrivee": "statut_a_l_arrivee",
            "Date_arrivee_malade": "date_arrivee_malade", "Date_admission_au_CT": "date_admission_au_ct",
            "Date_notification": "date_notification", "Date_investigation": "date_investigation",
            "Date_debut_maladie": "date_debut_maladie", "Province_notification": "province_notification",
            "Zone_de_sante_notification": "zone_de_sante_notification",
            "Aire_de_sante_notification": "aire_de_sante_notification", "Semaine_epid": "semaine_epid",
            "Num_semaine_epid": "num_semaine_epid", "Annee_epid": "annee_epid", "Nom_complet": "nom_complet",
            "Sexe": "sexe", "Age_annee": "age_annee", "Age_mois": "age_mois", "Age": "age",
            "Unite_age": "unite_age", "Age_en_ans": "age_en_ans", "Tranche_age": "tranche_age",
            "Tranche_age_en_ans": "tranche_age_en_ans", "Profession": "profession",
            "Province_provenance": "province_provenance", "Zone_de_sante_provenance": "zone_de_sante_provenance",
            "Aire_de_sante_provenance": "aire_de_sante_provenance", "Adresse": "adresse",
            "Symptomes": "symptomes", "Prise_antibiotique_avant_admission": "prise_antibiotique_avant_admission",
            "Nom_antibiotique": "nom_antibiotique", "Antecedents_morbides": "antecedents_morbides",
            "Femme_enceinte": "femme_enceinte", "Degre_deshydratation": "degre_deshydratation",
            "Plan_de_deshydratation": "plan_de_deshydratation", "Hospitalisation": "hospitalisation",
            "Prelevement": "prelevement", "Date_prelevement": "date_prelevement", "TDR_realise": "tdr_realise",
            "TDR_Resultat": "tdr_resultat", "TDR_archive": "tdr_archive", "Resultat_labo": "resultat_labo",
            "Resultat_labo_culture": "resultat_labo_culture", "Serotype": "serotype",
            "Nom_structure_realisant_le_tdr": "nom_structure_realisant_le_tdr", "Resultat_labo_pcr": "resultat_labo_pcr",
            "Traitement_antibiotique": "traitement_antibiotique", "Quantite_total_ringer_recue": "quantite_total_ringer_recue",
            "Quantite_total_sro_recue": "quantite_total_sro_recue", "Ctc_utc": "ctc_utc", "Issue": "issue",
            "Date_sortie_au_CT": "date_sortie_au_ct", "Etat_sortie_malade": "etat_sortie_malade",
            "Statut_vaccinal": "statut_vaccinal", "Nombre_dose": "nombre_dose",
            "Annee_vaccination": "annee_vaccination",
            "Source_eventuelle_de_contamination": "source_eventuelle_de_contamination",
            "Source_approvisionnement_en_eau": "source_approvisionnement_en_eau",
            "Classification_finale": "classification_finale", "Date_de_guerie": "date_de_guerie",
            "Observation": "observation", "est_cas_suspect": "est_cas_suspect",
            "est_cas_confirme": "est_cas_confirme", "classification_auto": "classification_auto"
        }
        df = df.rename(columns=rename_map)
        dfs.append(df)
    if not dfs:
        raise ValueError("Aucun DataFrame LL chargé")
    return pd.concat(dfs, ignore_index=True)

ids_df = load_ids()
ll_df = load_ll()
logger.info("IDS rows: %d, LL rows: %d", len(ids_df), len(ll_df))


## Nettoyer et transformer les données
- Appliquer les transformations de colonnes
- Nettoyer les valeurs manquantes
- Préparer les colonnes pour la base de données

In [ ]:
def normalize_text_columns(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip().replace({"": None})
    return df

ids_df = normalize_text_columns(ids_df, ["code_zone", "pays", "province", "zone_sante", "maladie"])
ll_df = normalize_text_columns(ll_df, [
    "n_epid_prov", "n_epid", "statut_a_l_arrivee", "province_notification",
    "zone_de_sante_notification", "aire_de_sante_notification", "nom_complet",
    "sexe", "unite_age", "tranche_age", "tranche_age_en_ans", "profession",
    "province_provenance", "zone_de_sante_provenance", "aire_de_sante_provenance",
    "adresse", "symptomes", "prise_antibiotique_avant_admission", "nom_antibiotique",
    "antecedents_morbides", "femme_enceinte", "degre_deshydratation",
    "plan_de_deshydratation", "hospitalisation", "prelevement", "tdr_realise",
    "tdr_resultat", "tdr_archive", "resultat_labo", "resultat_labo_culture",
    "serotype", "nom_structure_realisant_le_tdr", "resultat_labo_pcr",
    "traitement_antibiotique", "ctc_utc", "issue", "etat_sortie_malade",
    "statut_vaccinal", "source_eventuelle_de_contamination",
    "source_approvisionnement_en_eau", "classification_finale", "classification_auto"
])

int_cols = [
    "population", "num_semaine", "cas_tnn", "deces_tnn", "cas_0_11_mois", "deces_0_11_mois",
    "cas_12_59_mois", "deces_12_59_mois", "cas_5_15_ans", "deces_5_15_ans",
    "cas_15_plus", "deces_15_plus", "cas_total", "deces_total", "rec_status", "unique_key"
]
for c in int_cols:
    if c in ids_df.columns:
        ids_df[c] = pd.to_numeric(ids_df[c], errors="coerce").fillna(0).astype("Int64")

for d in [
    "date_arrivee_malade", "date_admission_au_ct", "date_notification", "date_investigation",
    "date_debut_maladie", "date_prelevement", "date_sortie_au_ct", "date_de_guerie"
]:
    if d in ll_df.columns:
        ll_df[d] = pd.to_datetime(ll_df[d], errors="coerce")

for n in ["num_semaine_epid", "annee_epid", "nombre_dose", "quantite_total_ringer_recue", "quantite_total_sro_recue"]:
    if n in ll_df.columns:
        ll_df[n] = pd.to_numeric(ll_df[n], errors="coerce")

ids_df = ids_df.where(pd.notnull(ids_df), None)
ll_df = ll_df.where(pd.notnull(ll_df), None)
logger.info("Nettoyage et transformation terminés")


## Préparer les données avant insertion
- Vérifier les types de données
- Convertir les formats
- Valider les doublons et le contenu avant insertion

In [ ]:
def validate_ids(df):
    required = ["code_zone", "num_semaine", "annee", "maladie"]
    missing = {col: int(df[col].isna().sum()) for col in required if col in df.columns}
    duplicates = int(df.duplicated(subset=["code_zone", "num_semaine", "maladie"]).sum()) if set(["code_zone", "num_semaine", "maladie"]).issubset(df.columns) else 0
    return missing, duplicates

def validate_ll(df):
    required = ["n_epid", "province_notification", "num_semaine_epid", "annee_epid"]
    missing = {col: int(df[col].isna().sum()) for col in required if col in df.columns}
    duplicates = int(df.duplicated(subset=["n_epid", "province_notification", "num_semaine_epid", "annee_epid"]).sum()) if set(["n_epid", "province_notification", "num_semaine_epid", "annee_epid"]).issubset(df.columns) else 0
    return missing, duplicates

ids_missing, ids_dups = validate_ids(ids_df)
ll_missing, ll_dups = validate_ll(ll_df)
print("IDS missing:", ids_missing)
print("IDS duplicates:", ids_dups)
print("LL missing:", ll_missing)
print("LL duplicates:", ll_dups)


## Connexion à PostgreSQL
Créer une connexion SQLAlchemy et configurer le moteur de base de données.

In [ ]:
def get_engine():
    user = os.environ["POSTGRES_USER"]
    password = os.environ["POSTGRES_PASSWORD"]
    db = os.environ["POSTGRES_DB"]
    host = os.environ["POSTGRES_HOST"]
    port = os.environ["POSTGRES_PORT"]
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
    return create_engine(url)

engine = get_engine()
logger.info("Connexion PostgreSQL configurée")


## Insérer les données dans PostgreSQL
Créer le schéma si nécessaire, puis insérer les données préparées dans les tables de destination.

In [ ]:
# Déduplication avant insertion

def deduplicate_ids(df):
    """Supprime les doublons dans les données IDS.
    Stratégie : si la colonne 'unique_key' existe, on garde la ligne avec la valeur maximale,
    sinon on garde la première occurrence selon le sous-ensemble clé.
    """
    before = len(df)
    subset = ["code_zone", "num_semaine", "maladie"]
    if 'unique_key' in df.columns:
        df = df.sort_values(by='unique_key', ascending=False)
    df = df.drop_duplicates(subset=subset, keep='first')
    removed = before - len(df)
    logger.info('IDS: removed %d duplicate rows', removed)
    return df


def deduplicate_ll(df):
    """Supprime les doublons dans les données LL.
    Stratégie : si 'date_notification' existe, on garde la ligne la plus récente,
    sinon on garde la première occurrence selon le sous-ensemble clé.
    """
    before = len(df)
    subset = ["n_epid", "province_notification", "num_semaine_epid", "annee_epid"]
    if 'date_notification' in df.columns:
        df = df.sort_values(by='date_notification', ascending=False)
    df = df.drop_duplicates(subset=subset, keep='first')
    removed = before - len(df)
    logger.info('LL: removed %d duplicate rows', removed)
    return df


# Appliquer la déduplication avant d'insérer en base
ids_df = deduplicate_ids(ids_df)
ll_df = deduplicate_ll(ll_df)
print('After deduplication - IDS rows:', len(ids_df), 'LL rows:', len(ll_df))


In [ ]:
def ensure_schema(engine):
    init_file = DB_DIR / "init.sql"
    if not init_file.exists():
        raise FileNotFoundError(f"Fichier de schéma introuvable: {init_file}")
    sql = init_file.read_text(encoding="utf-8")
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)

ensure_schema(engine)
logger.info("Schéma relationnel créé ou vérifié")

with engine.begin() as conn:
    ids_df.to_sql("cas_maladie", conn, schema="cholera", if_exists="append", index=False, method="multi")
    ll_df.to_sql("cas_ll", conn, schema="cholera", if_exists="append", index=False, method="multi")
logger.info("Importation SQL terminée")


## Vérifier l'insertion
Exécuter des requêtes de sélection pour confirmer que les données ont bien été insérées.

In [ ]:
with engine.connect() as conn:
    ids_count = conn.execute(text("SELECT count(*) FROM cholera.cas_maladie")).scalar()
    ll_count = conn.execute(text("SELECT count(*) FROM cholera.cas_ll")).scalar()
    print("Rows in cholera.cas_maladie:", ids_count)
    print("Rows in cholera.cas_ll:", ll_count)
    sample_ids = conn.execute(text("SELECT code_zone, num_semaine, maladie, pays, province, zone_sante FROM cholera.cas_maladie LIMIT 5")).fetchall()
    sample_ll = conn.execute(text("SELECT n_epid, province_notification, zone_de_sante_notification, sexe, issue FROM cholera.cas_ll LIMIT 5")).fetchall()
    print("Sample IDS:", sample_ids)
    print("Sample LL:", sample_ll)
